# 01 - Análisis Exploratorio de Datos (EDA)

## Enron Email Dataset

Este notebook tiene como objetivo realizar un análisis exploratorio inicial del conjunto de datos Enron Email Dataset.

El análisis permitirá comprender la estructura y calidad de los datos antes de abordar las etapas de preprocesamiento, extracción de información y modelado mediante técnicas de Procesamiento del Lenguaje Natural (NLP) y Machine Learning.

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Versión de Python:", sys.version)
print("Entorno utilizado:", sys.executable)

Versión de Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Entorno utilizado: c:\Users\Usuario\Documents\UCM Master\TFM Ana Valeria\.venv\Scripts\python.exe


## 1. Carga inicial del dataset

En primer lugar, se realizará una carga parcial del dataset para inspeccionar su estructura sin necesidad de cargar inicialmente el conjunto completo de correos electrónicos en memoria.

In [2]:
ruta_datos = "../data/raw/emails.csv"

df_sample = pd.read_csv(ruta_datos, nrows=1000)

df_sample.head()

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


## 2. Inspección de la estructura de los mensajes

El dataset contiene dos variables principales: `file`, que identifica la ubicación del mensaje dentro del corpus, y `message`, que almacena el contenido completo del correo electrónico en formato de texto.

Se inspeccionará un mensaje completo para identificar los distintos campos disponibles y determinar qué información puede extraerse posteriormente de forma estructurada.

In [8]:
print(df_sample["message"].iloc[0])

Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 


## 3. Análisis inicial de la muestra

A continuación, se analiza la dimensión de la muestra y la calidad inicial de las variables disponibles, prestando especial atención a la presencia de valores nulos y posibles registros duplicados.

In [9]:
print("Número de filas:", df_sample.shape[0])
print("Número de columnas:", df_sample.shape[1])

print("\nColumnas disponibles:")
print(df_sample.columns.tolist())

print("\nValores nulos:")
print(df_sample.isnull().sum())

print("\nFilas duplicadas:")
print(df_sample.duplicated().sum())

Número de filas: 1000
Número de columnas: 2

Columnas disponibles:
['file', 'message']

Valores nulos:
file       0
message    0
dtype: int64

Filas duplicadas:
0


## 4. Extracción de campos estructurados

Los mensajes se encuentran almacenados como texto sin estructurar, incluyendo tanto las cabeceras del correo como su contenido.

Para facilitar el análisis posterior, se utilizará el módulo `email` de Python para extraer los principales campos de cada mensaje: identificador, fecha, remitente, destinatario, asunto y cuerpo del correo.

In [10]:
from email import policy
from email.parser import Parser

mensaje_ejemplo = Parser(policy=policy.default).parsestr(
    df_sample["message"].iloc[0]
)

print("Message-ID:", mensaje_ejemplo["Message-ID"])
print("Date:", mensaje_ejemplo["Date"])
print("From:", mensaje_ejemplo["From"])
print("To:", mensaje_ejemplo["To"])
print("Subject:", mensaje_ejemplo["Subject"])

print("\nBody:")
print(mensaje_ejemplo.get_body(preferencelist=("plain",)).get_content())

Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 

Body:
Here is our forecast

 


## 5. Transformación de los mensajes en variables estructuradas

Una vez comprobado que el módulo `email` permite interpretar correctamente los mensajes, se define una función para extraer automáticamente los principales campos de cada correo electrónico.

Se conservarán inicialmente el identificador del mensaje, la fecha, el remitente, el destinatario, el asunto y el cuerpo del mensaje.

In [11]:
from email import policy
from email.parser import Parser

def parse_email(raw_message):
    mensaje = Parser(policy=policy.default).parsestr(raw_message)

    # Extraer el cuerpo del correo
    body_part = mensaje.get_body(preferencelist=("plain",))

    if body_part is not None:
        body = body_part.get_content()
    else:
        body = ""

    return {
        "message_id": mensaje["Message-ID"],
        "date": mensaje["Date"],
        "from": mensaje["From"],
        "to": mensaje["To"],
        "subject": mensaje["Subject"],
        "body": body
    }

In [12]:
parse_email(df_sample["message"].iloc[0])

{'message_id': '<18782981.1075855378110.JavaMail.evans@thyme>',
 'date': 'Mon, 14 May 2001 16:39:00 -0700',
 'from': 'phillip.allen@enron.com',
 'to': 'tim.belden@enron.com',
 'subject': '',
 'body': 'Here is our forecast\n\n '}

## 6. Creación del dataset estructurado

La función de extracción se aplica a todos los mensajes de la muestra para transformar el contenido original en un conjunto de datos estructurado.

Este proceso permitirá trabajar de forma independiente con los metadatos de los correos y con su contenido textual, facilitando las posteriores etapas de análisis exploratorio, limpieza y procesamiento del lenguaje natural.

In [13]:
emails_parsed = df_sample["message"].apply(parse_email)

df_emails = pd.DataFrame(emails_parsed.tolist())

df_emails.head()

,message_id,date,from,to,subject,body
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast\n\n
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 04 May 2001 13:51:00 -0700",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


In [14]:
print("Dimensiones:", df_emails.shape)

print("\nColumnas:")
print(df_emails.columns.tolist())

print("\nValores nulos:")
print(df_emails.isnull().sum())

Dimensiones: (1000, 6)

Columnas:
['message_id', 'date', 'from', 'to', 'subject', 'body']

Valores nulos:
message_id    0
date          0
from          0
to            5
subject       0
body          0
dtype: int64


## 7. Análisis de valores ausentes y campos vacíos

Además de los valores nulos, es necesario identificar campos almacenados como cadenas de texto vacías, ya que estos no son detectados mediante el método `isnull()`.

Este análisis resulta especialmente relevante para los campos `subject` y `body`, cuyo contenido tendrá un papel importante en las posteriores etapas de procesamiento de lenguaje natural.

In [15]:
for columna in ["from", "to", "subject", "body"]:
    nulos = df_emails[columna].isnull().sum()

    vacios = (
        df_emails[columna]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{columna}:")
    print(f"  Nulos: {nulos}")
    print(f"  Vacíos o nulos: {vacios}")

from:
  Nulos: 0
  Vacíos o nulos: 0
to:
  Nulos: 5
  Vacíos o nulos: 5
subject:
  Nulos: 0
  Vacíos o nulos: 302
body:
  Nulos: 0
  Vacíos o nulos: 0


## 8. Análisis inicial de duplicados

Se analiza la existencia de posibles mensajes duplicados utilizando el identificador único del correo (`Message-ID`) y el contenido textual del mensaje.

In [16]:
print("Message-ID duplicados:")
print(df_emails["message_id"].duplicated().sum())

print("\nCuerpos de correo duplicados:")
print(df_emails["body"].duplicated().sum())

Message-ID duplicados:
0

Cuerpos de correo duplicados:
343


## 9. Inspección de mensajes con contenido duplicado

La existencia de cuerpos de correo duplicados no implica necesariamente que se trate de registros completamente duplicados. Diferentes mensajes pueden compartir el mismo contenido, pero presentar distintos identificadores, destinatarios, fechas u otros metadatos.

Por este motivo, antes de aplicar cualquier estrategia de eliminación de duplicados, se inspeccionarán los mensajes afectados para determinar su naturaleza.

In [17]:
duplicados_body = df_emails[
    df_emails.duplicated(subset=["body"], keep=False)
].sort_values("body")

print("Correos implicados en contenidos duplicados:", len(duplicados_body))

duplicados_body[
    ["message_id", "date", "from", "to", "subject", "body"]
].head(20)

Correos implicados en contenidos duplicados: 686


,message_id,date,from,to,subject,body
53,<22955100.1075855688377.JavaMail.evans@thyme>,"Mon, 11 Sep 2000 08:07:00 -0700",phillip.allen@enron.com,john.lavorato@enron.com,,"9/8 9/7 diff\n\nSocal 36,600 37,200 -60..."
642,<21095120.1075855668506.JavaMail.evans@thyme>,"Mon, 11 Sep 2000 08:07:00 -0700",phillip.allen@enron.com,john.lavorato@enron.com,,"9/8 9/7 diff\n\nSocal 36,600 37,200 -60..."
825,<31380481.1075855672099.JavaMail.evans@thyme>,"Mon, 06 Mar 2000 05:27:00 -0800",phillip.allen@enron.com,maryrichards7@hotmail.com,Re:,$100 for the yard seems like enough for up to ...
231,<18916002.1075855691857.JavaMail.evans@thyme>,"Mon, 06 Mar 2000 05:27:00 -0800",phillip.allen@enron.com,maryrichards7@hotmail.com,Re:,$100 for the yard seems like enough for up to ...
263,<12222503.1075855692514.JavaMail.evans@thyme>,"Tue, 18 Jan 2000 10:06:00 -0800",phillip.allen@enron.com,pallen70@hotmail.com,RE: Choosing a style,---------------------- Forwarded by Phillip K ...
861,<33120873.1075855672841.JavaMail.evans@thyme>,"Tue, 18 Jan 2000 10:06:00 -0800",phillip.allen@enron.com,pallen70@hotmail.com,RE: Choosing a style,---------------------- Forwarded by Phillip K ...
860,<32075032.1075855672819.JavaMail.evans@thyme>,"Thu, 27 Jan 2000 08:44:00 -0800",phillip.allen@enron.com,"fletcher.sturm@enron.com, hunter.shively@enron...",dopewars,---------------------- Forwarded by Phillip K ...
262,<3189067.1075855692492.JavaMail.evans@thyme>,"Thu, 27 Jan 2000 08:44:00 -0800",phillip.allen@enron.com,"fletcher.sturm@enron.com, hunter.shively@enron...",dopewars,---------------------- Forwarded by Phillip K ...
255,<23458569.1075855692364.JavaMail.evans@thyme>,"Fri, 04 Feb 2000 09:09:00 -0800",phillip.allen@enron.com,pallen70@hotmail.com,,---------------------- Forwarded by Phillip K ...
853,<14732638.1075855672690.JavaMail.evans@thyme>,"Fri, 04 Feb 2000 09:09:00 -0800",phillip.allen@enron.com,pallen70@hotmail.com,,---------------------- Forwarded by Phillip K ...


In [18]:
frecuencia_body = (
    df_emails
    .groupby("body")
    .size()
    .sort_values(ascending=False)
)

frecuencia_body.head(10)

body
re: window unit check with gary about what kind he wants to install                                                                                                                                                                                       2
received the file.  It worked.  Good job.                                                                                                                                                                                                                 2
resumes of whom?                                                                                                                                                                                                                                          2
socal position\n\n\n\n\nThis is short, but is it good enough?\n                                                                                                                                                                                

### Conclusiones sobre los duplicados

El análisis de la muestra revela una presencia relevante de contenidos duplicados. Aunque no se detectan identificadores `Message-ID` repetidos, 686 de los 1.000 mensajes analizados pertenecen a grupos cuyo cuerpo aparece más de una vez.

La inspección de estos casos muestra que diferentes mensajes pueden presentar cuerpos idénticos aun disponiendo de identificadores únicos. Por tanto, el `Message-ID` no resulta suficiente para detectar la redundancia existente en el corpus.

Este aspecto deberá considerarse durante la fase de preprocesamiento y, especialmente, antes de dividir los datos en conjuntos de entrenamiento y evaluación, con el objetivo de evitar que mensajes con contenido idéntico aparezcan en ambos conjuntos y puedan producir fuga de información (data leakage).

### Consideraciones sobre la muestra analizada

La muestra inicial de 1.000 registros se ha utilizado con un objetivo exploratorio y técnico, principalmente para comprender la estructura de los datos, validar el proceso de extracción de los campos de los correos y realizar las primeras comprobaciones de calidad.

No obstante, esta muestra no puede considerarse representativa del conjunto completo de datos, ya que las primeras filas del dataset se encuentran agrupadas por usuario y corresponden mayoritariamente a correos asociados a Phillip Allen. Por este motivo, las estadísticas obtenidas en esta fase, como la proporción de asuntos vacíos o la presencia de cuerpos duplicados, no deben extrapolarse al conjunto completo.

El análisis definitivo de la distribución, calidad y características del corpus se realizará posteriormente sobre el dataset completo.

# Análisis del dataset completo

Una vez validada la estructura de los mensajes y el procedimiento de extracción sobre una muestra inicial de 1.000 registros, se procede al análisis del conjunto completo de datos.

A partir de este punto, las estadísticas obtenidas se calcularán sobre el corpus completo, permitiendo obtener una visión representativa de su estructura, calidad y principales características.

In [3]:
df = pd.read_csv(ruta_datos)

print("Número de registros:", df.shape[0])
print("Número de columnas:", df.shape[1])
print("\nColumnas:", df.columns.tolist())

Número de registros: 517401
Número de columnas: 2

Columnas: ['file', 'message']


## 10. Comprobación inicial del corpus completo

El dataset completo contiene 517.401 registros y dos variables originales: `file` y `message`. Antes de proceder con la extracción de los campos internos de los mensajes, se comprueba la existencia de valores nulos y registros duplicados en su estructura original.

In [20]:
print("Valores nulos:")
print(df.isnull().sum())

print("\nFilas completamente duplicadas:")
print(df.duplicated().sum())

print("\nArchivos duplicados:")
print(df["file"].duplicated().sum())

print("\nMensajes completos duplicados:")
print(df["message"].duplicated().sum())

Valores nulos:
file       0
message    0
dtype: int64

Filas completamente duplicadas:
0

Archivos duplicados:
0

Mensajes completos duplicados:
0


Una vez validado el procedimiento de extracción sobre la muestra inicial, se aplica el parser desarrollado al conjunto completo de mensajes para obtener una representación estructurada de los correos electrónicos.

In [5]:
#emails_parsed = df["message"].apply(parse_email)
#df_emails_full = pd.DataFrame(emails_parsed.tolist())
#print("Dimensiones del dataset estructurado:", df_emails_full.shape)
#df_emails_full.head()

## 11. Optimización del procesamiento del corpus

Tras validar el parser sobre la muestra inicial, se evaluó su aplicación sobre el conjunto completo de 517.401 mensajes. Sin embargo, el procesamiento secuencial mediante el parser estándar presentó un elevado coste computacional.

Por este motivo, se optó por desarrollar un procedimiento de extracción más ligero, orientado exclusivamente a los campos necesarios para el análisis. Antes de aplicarlo al corpus completo, el nuevo procedimiento se valida sobre un subconjunto de 10.000 mensajes.

In [8]:
def extraer_campos_rapido(raw_message):
    # Separar cabeceras y cuerpo
    partes = raw_message.split("\n\n", 1)

    headers = partes[0]
    body = partes[1] if len(partes) > 1 else ""

    def extraer_header(nombre):
        prefijo = nombre + ":"

        for linea in headers.splitlines():
            if linea.lower().startswith(prefijo.lower()):
                return linea[len(prefijo):].strip()

        return None

    return {
        "message_id": extraer_header("Message-ID"),
        "date": extraer_header("Date"),
        "from": extraer_header("From"),
        "to": extraer_header("To"),
        "subject": extraer_header("Subject"),
        "body": body.strip()
    }

In [9]:
prueba_10k = df["message"].head(10000).apply(extraer_campos_rapido)

df_prueba_10k = pd.DataFrame(prueba_10k.tolist())

print("Dimensiones:", df_prueba_10k.shape)

df_prueba_10k.head()

Dimensiones: (10000, 6)


,message_id,date,from,to,subject,body
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


### Aplicación del procedimiento optimizado al corpus completo

Tras comprobar el correcto funcionamiento del procedimiento de extracción sobre un subconjunto de 10.000 mensajes, se aplica la función optimizada al corpus completo de 517.401 correos electrónicos.

El resultado será un conjunto de datos estructurado en seis variables: identificador del mensaje, fecha, remitente, destinatario, asunto y cuerpo del correo.

In [10]:
emails_parsed_full = df["message"].apply(extraer_campos_rapido)

df_emails_full = pd.DataFrame(emails_parsed_full.tolist())

print("Dimensiones del dataset estructurado:", df_emails_full.shape)

df_emails_full.head()

Dimensiones del dataset estructurado: (517401, 6)


,message_id,date,from,to,subject,body
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


## 12. Control de calidad del corpus estructurado

Una vez estructurado el conjunto completo de correos electrónicos, se realiza un análisis de calidad de los datos. Se examinan los valores nulos o vacíos de las variables extraídas, así como la existencia de identificadores y contenidos duplicados.

Este análisis permitirá determinar qué decisiones de limpieza serán necesarias antes de abordar las etapas posteriores de procesamiento del texto y modelado.

In [11]:
print("=== VALORES NULOS ===")
print(df_emails_full.isnull().sum())

print("\n=== VALORES VACÍOS O NULOS ===")

for columna in ["from", "to", "subject", "body"]:
    nulos = df_emails_full[columna].isnull().sum()
    vacios = df_emails_full[columna].fillna("").str.strip().eq("").sum()

    print(f"{columna}:")
    print(f"  Nulos: {nulos}")
    print(f"  Vacíos o nulos: {vacios}")

print("\n=== DUPLICADOS ===")

print(
    "Message-ID duplicados:",
    df_emails_full["message_id"].duplicated().sum()
)

print(
    "Cuerpos de correo duplicados:",
    df_emails_full["body"].duplicated().sum()
)

=== VALORES NULOS ===
message_id        0
date              0
from              0
to            21847
subject           0
body              0
dtype: int64

=== VALORES VACÍOS O NULOS ===
from:
  Nulos: 0
  Vacíos o nulos: 0
to:
  Nulos: 21847
  Vacíos o nulos: 22427
subject:
  Nulos: 0
  Vacíos o nulos: 19187
body:
  Nulos: 0
  Vacíos o nulos: 0

=== DUPLICADOS ===
Message-ID duplicados: 0
Cuerpos de correo duplicados: 269529


### Análisis de los contenidos duplicados

El análisis de calidad muestra que no existen identificadores de mensaje duplicados, lo que indica que cada registro corresponde a un mensaje identificado de forma independiente dentro del corpus.

Sin embargo, se observa un número elevado de cuerpos de correo repetidos. La coincidencia del contenido textual no implica necesariamente que se trate de registros duplicados, ya que un mismo contenido puede aparecer asociado a diferentes mensajes, remitentes, destinatarios o ubicaciones dentro del corpus.

Por este motivo, antes de aplicar cualquier estrategia de eliminación de duplicados, se analiza la naturaleza de estas repeticiones.

In [12]:
duplicados_body = df_emails_full[
    df_emails_full.duplicated(subset=["body"], keep=False)
].sort_values("body")

print("Correos implicados en contenidos repetidos:", len(duplicados_body))

print(
    "Cuerpos de correo diferentes que aparecen más de una vez:",
    df_emails_full.loc[
        df_emails_full["body"].duplicated(keep=False),
        "body"
    ].nunique()
)

duplicados_body[
    ["message_id", "date", "from", "to", "subject", "body"]
].head(20)

Correos implicados en contenidos repetidos: 401575
Cuerpos de correo diferentes que aparecen más de una vez: 132046


,message_id,date,from,to,subject,body
187873,<2286948.1075847309995.JavaMail.evans@thyme>,"Tue, 27 Feb 2001 05:35:00 -0800 (PST)",tana.jones@enron.com,enron.security@enron.com,Re: eRequests 20721 & 20740,"!. I'll be trying to do that big request, and..."
197679,<7562145.1075847549189.JavaMail.evans@thyme>,"Tue, 27 Feb 2001 05:35:00 -0800 (PST)",tana.jones@enron.com,enron.security@enron.com,Re: eRequests 20721 & 20740,"!. I'll be trying to do that big request, and..."
77724,<26550855.1075843113090.JavaMail.evans@thyme>,"Tue, 26 Sep 2000 09:06:00 -0700 (PDT)",susan.mara@enron.com,jeff.dasovich@enron.com,Re: Technical Assistance for Transmission,""" I was a lineman for the county...""\n--------..."
59677,<14792206.1075842980578.JavaMail.evans@thyme>,"Tue, 26 Sep 2000 09:06:00 -0700 (PDT)",susan.mara@enron.com,jeff.dasovich@enron.com,Re: Technical Assistance for Transmission,""" I was a lineman for the county...""\n--------..."
403851,<11897262.1075846818585.JavaMail.evans@thyme>,"Tue, 18 Apr 2000 07:26:00 -0700 (PDT)",susan.scott@enron.com,sunil.dalal@enron.com,A barrage of Inspiration,""" To dream anything that you want to dream -\n..."
407827,<8241973.1075846770162.JavaMail.evans@thyme>,"Tue, 18 Apr 2000 07:26:00 -0700 (PDT)",susan.scott@enron.com,sunil.dalal@enron.com,A barrage of Inspiration,""" To dream anything that you want to dream -\n..."
404817,<8639342.1075846763014.JavaMail.evans@thyme>,"Tue, 18 Apr 2000 07:26:00 -0700 (PDT)",susan.scott@enron.com,sunil.dalal@enron.com,A barrage of Inspiration,""" To dream anything that you want to dream -\n..."
410742,<29885614.1075846808002.JavaMail.evans@thyme>,"Tue, 18 Apr 2000 07:26:00 -0700 (PDT)",susan.scott@enron.com,sunil.dalal@enron.com,A barrage of Inspiration,""" To dream anything that you want to dream -\n..."
275523,<24175583.1075849755986.JavaMail.evans@thyme>,"Tue, 3 Apr 2001 06:23:00 -0700 (PDT)",matthew.lenhart@enron.com,shirley.s.elliott@citicorp.com,RE:,""" i know i'm pretty but i ain't as pretty as a..."
279387,<27271765.1075849784504.JavaMail.evans@thyme>,"Tue, 3 Apr 2001 06:23:00 -0700 (PDT)",matthew.lenhart@enron.com,shirley.s.elliott@citicorp.com,RE:,""" i know i'm pretty but i ain't as pretty as a..."


In [13]:
frecuencia_body = (
    df_emails_full["body"]
    .value_counts()
    .reset_index()
)

frecuencia_body.columns = ["body", "frecuencia"]

frecuencia_body.head(20)

,body,frecuencia
0,"As you know, Enron Net Works (ENW) and Enron G...",112
1,We've updated the Merger Q&A document on our E...,110
2,Please see attached.,108
3,Ken Lay and Jeff Skilling were interviewed on ...,107
4,Our natural gas business continues to benefit ...,107
5,"As you know, this is an unprecedented time in ...",106
6,eSource Presents Lexis-Nexis Training\n\nBasic...,105
7,Today we announced the appointment of Jeff McM...,103
8,"As you know, Enron, its directors, and certain...",101
9,It is my great pleasure to announce that the B...,101


# Conclusiones del análisis y adecuación del dataset al proyecto

El análisis exploratorio realizado sobre el **Enron Email Dataset** ha permitido estudiar la estructura de un corpus real de correos electrónicos y desarrollar un procedimiento para extraer de los mensajes información estructurada como el identificador, la fecha, el remitente, el destinatario, el asunto y el cuerpo del correo.

El dataset contiene **517.401 mensajes**, lo que proporciona un volumen de información suficiente para desarrollar técnicas de procesamiento de texto. Durante el análisis también se identificó una presencia elevada de contenidos repetidos, así como determinados valores ausentes en campos como el destinatario o el asunto.

Sin embargo, el análisis del contenido de los mensajes permitió identificar una limitación relevante respecto al objetivo del Trabajo Fin de Máster. El corpus de Enron está formado principalmente por comunicaciones vinculadas a la actividad interna de la organización y no constituye específicamente un conjunto de consultas realizadas por clientes a una empresa.

Esta característica dificulta la definición de variables objetivo directamente relacionadas con la clasificación y gestión de consultas de clientes. En particular, el dataset no proporciona etiquetas de negocio adecuadas que permitan identificar de forma supervisada el tipo de consulta, el producto relacionado o el motivo del contacto.

Por este motivo, aunque el dataset resulta adecuado para experimentar con técnicas generales de procesamiento de lenguaje natural, se considera que **no es la fuente de datos más apropiada para el objetivo específico del proyecto**.

A partir de los resultados obtenidos en esta fase exploratoria, se decide evaluar un conjunto de datos alternativo que contenga comunicaciones realizadas directamente por clientes y que disponga de categorías asociadas a dichas comunicaciones. Esta decisión permitirá alinear mejor los datos disponibles con el problema de clasificación automática planteado en el TFM.

El análisis realizado sobre Enron se conserva como parte del proyecto, ya que documenta el proceso de evaluación y selección de la fuente de datos y permitió, además, validar procedimientos de carga, extracción y estructuración de información textual que podrán reutilizarse en las siguientes fases.

## Siguiente etapa del proyecto

Como consecuencia de las limitaciones identificadas, la siguiente fase del proyecto analizará la **Consumer Complaint Database del Consumer Financial Protection Bureau (CFPB)**.

Este conjunto de datos contiene comunicaciones de consumidores relacionadas con diferentes productos y servicios financieros, junto con variables categóricas como el producto, el problema y el subproblema asociado a cada reclamación.

La disponibilidad simultánea de **texto libre y categorías previamente definidas** permitirá evaluar la construcción de modelos supervisados de clasificación de texto y estudiar su aplicación a un sistema de gestión automática de consultas de clientes.